In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import datetime
import glob
import hashlib
import os
from urllib.request import urlretrieve

In [ ]:
# Verify env variables
import os
print(os.environ.get('START_YEAR'), os.environ.get('END_YEAR'), os.environ.get('MONTHS'))

In [2]:
def parse_csv_env(name, default):
    raw_value = os.environ.get(name, default)
    return [item.strip() for item in raw_value.split(',') if item.strip()]

services = parse_csv_env('SERVICES', 'yellow,green')
start_year = int(os.environ.get('START_YEAR', '2015'))
end_year = int(os.environ.get('END_YEAR', '2025'))
months = [int(item) for item in parse_csv_env('MONTHS', '1,2,3,4,5,6,7,8,9,10,11,12')]
run_id = os.environ.get('RUN_ID', datetime.datetime.now().strftime("%Y%m%d_%H%M%S"))

In [ ]:
def log_step(message):
    timestamp = datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')
    print(f'[{timestamp}Z] {message}')


def build_spark(app_name):
    jars_dir = os.environ.get('SPARK_JARS_DIR', '/home/jovyan/jars')
    jar_paths = sorted(glob.glob(os.path.join(jars_dir, '*.jar')))
    builder = (
        SparkSession.builder
        .appName(app_name)
        .config('spark.sql.session.timeZone', 'UTC')
        .config('spark.driver.memory', '64g')
        .config('spark.executor.memory', '32g')
        .config('spark.sql.shuffle.partitions', '200')
    )
    if jar_paths:
        jars_csv = ','.join(jar_paths)
        classpath = ':'.join(jar_paths)
        log_step(f'Loading JARs: {jar_paths}')
        builder = (
            builder
            .config('spark.jars', jars_csv)
            .config('spark.driver.extraClassPath', classpath)
            .config('spark.executor.extraClassPath', classpath)
        )
    else:
        log_step(f'No JARs found in {jars_dir}')
    return builder.getOrCreate()


def assert_snowflake_connector(spark):
    class_names = [
        'net.snowflake.spark.snowflake.DefaultSource',
        'snowflake.DefaultSource',
    ]
    for class_name in class_names:
        try:
            spark.sparkContext._jvm.java.lang.Class.forName(class_name)
            log_step(f'Snowflake connector detected: {class_name}')
            return
        except Exception:
            continue
    raise RuntimeError('Snowflake connector not found.')


app = build_spark('01_ingesta_parquet_raw')
assert_snowflake_connector(app)

# Credenciales Snowflake desde variables de ambiente
sf_option = {
    'sfURL': os.environ['SF_HOST'],
    'sfUser': os.environ['SF_USER'],
    'sfPassword': os.environ['SF_PASSWORD'],
    'sfDatabase': os.environ['SF_DATABASE'],
    'sfSchema': os.environ.get('SF_RAW_SCHEMA', 'raw'),
    'sfWarehouse': os.environ.get('SF_WAREHOUSE', ''),
    'sfRole': os.environ.get('SF_ROLE', ''),
}

# Helper to run arbitrary SQL on Snowflake
sf_utils = app._jvm.net.snowflake.spark.snowflake.Utils

def sf_run(sql):
    jsf = app._jvm.java.util.HashMap()
    for k, v in sf_option.items():
        jsf.put(k, v)
    sf_utils.runQuery(jsf, sql)

log_step(f'Run configured: services={services}, years={start_year}-{end_year}, months={months}, run_id={run_id}')

In [4]:
def get_nyc_url(service, year, month):
    base_url = os.environ.get(f'{service.upper()}_BASE_URL', 'https://d37ci6vzurychx.cloudfront.net/trip-data').rstrip('/')
    return f"{base_url}/{service}_tripdata_{year}-{month:02d}.parquet"

In [ ]:
# Scale warehouse even higher to speed up writes
sf_run('ALTER WAREHOUSE COMPUTE_WH SET WAREHOUSE_SIZE = XXLARGE')
log_step('Warehouse scaled to 2XLARGE')

In [5]:
def download_parquet(url, service, year, month):
    local_dir = os.environ.get('LOCAL_PARQUET_DIR', '/tmp/nyc_taxi_tripdata')
    os.makedirs(local_dir, exist_ok=True)
    local_path = os.path.join(local_dir, f'{service}_{year}_{month:02d}.parquet')
    if not os.path.exists(local_path):
        log_step(f'Downloading {url}')
        urlretrieve(url, local_path)
    else:
        log_step(f'Using cached parquet {local_path}')
    return local_path

In [6]:
def standardize_columns(df, target_columns):
    for field in df.schema.fields:
        if isinstance(field.dataType, TimestampNTZType):
            df = df.withColumn(field.name, F.col(field.name).cast(TimestampType()))

    for col_name, col_type in target_columns.items():
        if col_name in df.columns:
            df = df.withColumn(col_name, F.col(col_name).cast(col_type))
        else:
            df = df.withColumn(col_name, F.lit(None).cast(col_type))
    return df

In [7]:
def compute_trip_nk(df, service):
    columns = set(df.columns)

    def existing_col(*names):
        for name in names:
            if name in columns:
                return F.col(name)
        return F.lit(None)

    pickup_col = existing_col('tpep_pickup_datetime', 'lpep_pickup_datetime', 'pickup_datetime')
    dropoff_col = existing_col('tpep_dropoff_datetime', 'lpep_dropoff_datetime', 'dropoff_datetime')
    vendor_col = existing_col('VendorID', 'vendorid')
    pu_col = existing_col('PULocationID', 'pu_location_id')
    do_col = existing_col('DOLocationID', 'do_location_id')
    total_amount_col = existing_col('total_amount')

    return df.withColumn(
        'trip_nk',
        F.sha2(
            F.concat_ws(
                '||',
                F.lit(service),
                vendor_col.cast('string'),
                pickup_col.cast('string'),
                dropoff_col.cast('string'),
                pu_col.cast('string'),
                do_col.cast('string'),
                total_amount_col.cast('string')
            ),
            256
        )
    )

In [8]:
def log_audit(service, year, month, status, rows_read=0, rows_written=0, notes=""):
    audit_row = [(run_id, service, year, month, status, rows_read, rows_written, datetime.datetime.utcnow(), notes)]
    schema = "run_id string, service string, year int, month int, status string, rows_read long, rows_written long, event_timestamp timestamp, notes string"
    audit_df = app.createDataFrame(audit_row, schema)
    try:
        audit_df.write.format('snowflake').options(**sf_option).option('dbtable', 'RAW.LOAD_AUDIT').mode('append').save()
    except Exception as audit_error:
        log_step(f'Audit logging failed for service={service} year={year} month={month:02d}: {str(audit_error)[:200]}')

In [9]:
expected_schema = {
    'VendorID': IntegerType(),
    'tpep_pickup_datetime': TimestampType(),
    'tpep_dropoff_datetime': TimestampType(),
    'passenger_count': DoubleType(),
    'trip_distance': DoubleType(),
    'RatecodeID': DoubleType(),
    'PULocationID': IntegerType(),
    'DOLocationID': IntegerType(),
    'payment_type': LongType(),
    'fare_amount': DoubleType(),
    'total_amount': DoubleType()
}

In [ ]:
import pandas as pd
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
from concurrent.futures import ThreadPoolExecutor, as_completed

sf_run('ALTER WAREHOUSE COMPUTE_WH SET WAREHOUSE_SIZE = XLARGE')

# Drop existing tables so auto_create_table builds them with correct types
sf_run('DROP TABLE IF EXISTS RAW.YELLOW_TRIPS')
sf_run('DROP TABLE IF EXISTS RAW.GREEN_TRIPS')
sf_run('DROP TABLE IF EXISTS RAW.LOAD_AUDIT')
log_step('Dropped old tables, warehouse set to XLARGE')

def get_sf_connection():
    return snowflake.connector.connect(
        account=os.environ['SF_HOST'].replace('.snowflakecomputing.com', ''),
        user=os.environ['SF_USER'],
        password=os.environ['SF_PASSWORD'],
        database=os.environ['SF_DATABASE'],
        schema=os.environ.get('SF_RAW_SCHEMA', 'RAW'),
        warehouse=os.environ.get('SF_WAREHOUSE', ''),
        role=os.environ.get('SF_ROLE', ''),
    )

def process_batch_fast(service, year, month):
    url = get_nyc_url(service, year, month)
    target_table = f'{service.upper()}_TRIPS'
    local_path = f'/tmp/nyc_taxi_tripdata/{service}_{year}_{month:02d}.parquet'
    
    try:
        if not os.path.exists(local_path):
            log_step(f'Downloading {url}')
            urlretrieve(url, local_path)
        
        log_step(f'Reading {service} {year}-{month:02d}')
        df = pd.read_parquet(local_path)
        rows_read = len(df)
        log_step(f'Read {service} {year}-{month:02d}: {rows_read} rows')
        
        # Add metadata columns
        df['run_id'] = run_id
        df['service_type'] = service
        df['source_year'] = year
        df['source_month'] = month
        df['ingested_at_utc'] = pd.Timestamp.utcnow()
        df['source_path'] = url
        
        # Fast vectorized hash
        hash_cols = ['VendorID'] + [c for c in df.columns if 'pickup_datetime' in c or 'dropoff_datetime' in c] + ['total_amount']
        hash_cols = [c for c in hash_cols if c in df.columns]
        df['trip_nk'] = pd.util.hash_pandas_object(df[hash_cols], index=False).astype(str)
        
        # Uppercase all column names for Snowflake
        df.columns = [c.upper() for c in df.columns]
        
        # Write to Snowflake
        conn = get_sf_connection()
        try:
            raw_schema = os.environ.get('SF_RAW_SCHEMA', 'RAW')
            try:
                conn.cursor().execute(f"DELETE FROM {raw_schema}.{target_table} WHERE SOURCE_YEAR = {year} AND SOURCE_MONTH = {month}")
            except:
                pass
            
            log_step(f'Writing {service} {year}-{month:02d} ({rows_read} rows)')
            write_pandas(conn, df, target_table, schema=raw_schema, auto_create_table=True, overwrite=False, use_logical_type=True)
            
            log_step(f'OK {service} {year}-{month:02d} rows={rows_read}')
            return ('SUCCESS', service, year, month, rows_read)
        finally:
            conn.close()
    except Exception as e:
        status = 'Missing' if '403' in str(e) or '404' in str(e) or 'HTTP Error' in str(e) else 'ERROR'
        log_step(f'{status} {service} {year}-{month:02d}: {str(e)[:300]}')
        return (status, service, year, month, 0)

log_step('=== Fast load via pandas + write_pandas (2 parallel workers) ===')

all_tasks = []
for service in services:
    for year in range(start_year, end_year + 1):
        for month in months:
            all_tasks.append((service, year, month))

all_results = []
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(process_batch_fast, svc, yr, mo): (svc, yr, mo) for svc, yr, mo in all_tasks}
    for f in as_completed(futures):
        all_results.append(f.result())

success = sum(1 for r in all_results if r[0] == 'SUCCESS')
missing = sum(1 for r in all_results if r[0] == 'Missing')
errors = sum(1 for r in all_results if r[0] == 'ERROR')
log_step(f'=== Done! SUCCESS={success} Missing={missing} ERROR={errors} ===')

sf_run('ALTER WAREHOUSE COMPUTE_WH SET WAREHOUSE_SIZE = XSMALL')
log_step('Warehouse scaled back to XSMALL')

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def process_batch(service, year, month):
    """Process a single (service, year, month) batch."""
    url = get_nyc_url(service, year, month)
    target_table = f'{service.upper()}_TRIPS'
    try:
        log_step(f'Starting batch service={service} year={year} month={month:02d}')
        local_path = download_parquet(url, service, year, month)
        df_raw = app.read.parquet(local_path)
        df_raw = standardize_columns(df_raw, expected_schema)
        rows_read = df_raw.count()
        log_step(f'Rows read: {service} {year}-{month:02d} = {rows_read}')

        df_processed = (
            df_raw
            .withColumn('run_id', F.lit(run_id))
            .withColumn('service_type', F.lit(service))
            .withColumn('source_year', F.lit(year).cast('int'))
            .withColumn('source_month', F.lit(month).cast('int'))
            .withColumn('ingested_at_utc', F.current_timestamp())
            .withColumn('source_path', F.lit(url))
        )
        df_processed = compute_trip_nk(df_processed, service)

        raw_schema = os.environ.get('SF_RAW_SCHEMA', 'RAW')
        preactions = f'DELETE FROM {raw_schema}.{target_table} WHERE source_year = {year} AND source_month = {month}'

        log_step(f'Writing {service} {year}-{month:02d} to Snowflake')
        df_processed.write.format('snowflake').options(**sf_option).option('dbtable', target_table).option('preactions', preactions).mode('append').save()

        log_audit(service, year, month, "SUCCESS", rows_read, rows_read)
        log_step(f'OK {service} {year}-{month:02d} rows={rows_read}')
        return ('SUCCESS', service, year, month, rows_read)
    except Exception as e:
        status = 'Missing' if '403' in str(e) or '404' in str(e) or 'HTTP Error' in str(e) else 'ERROR'
        log_step(f'{status} {service} {year}-{month:02d}: {str(e)[:200]}')
        log_audit(service, year, month, status, 0, 0, notes=str(e)[:500])
        return (status, service, year, month, 0)

# Pre-download all parquet files in parallel first (network I/O)
log_step('=== Phase 1: Parallel download of all parquet files ===')
download_tasks = []
for service in services:
    for year in range(start_year, end_year + 1):
        for month in months:
            download_tasks.append((service, year, month))

def download_only(args):
    service, year, month = args
    url = get_nyc_url(service, year, month)
    try:
        local_path = download_parquet(url, service, year, month)
        return (True, service, year, month)
    except Exception as e:
        return (False, service, year, month)

with ThreadPoolExecutor(max_workers=16) as executor:
    dl_futures = {executor.submit(download_only, t): t for t in download_tasks}
    for f in as_completed(dl_futures):
        ok, svc, yr, mo = f.result()

log_step('=== Phase 2: Process and load to Snowflake ===')
# Process batches — Snowflake writes are sequential to avoid conflicts
results = []
for service in services:
    for year in range(start_year, end_year + 1):
        for month in months:
            result = process_batch(service, year, month)
            results.append(result)

success = sum(1 for r in results if r[0] == 'SUCCESS')
missing = sum(1 for r in results if r[0] == 'Missing')
errors = sum(1 for r in results if r[0] == 'ERROR')
log_step(f'=== Done! SUCCESS={success} Missing={missing} ERROR={errors} ===')

In [11]:
# Resumen del run actual
df_audit_summary = (
    app.read.format('snowflake').options(**sf_option)
    .option('dbtable', 'LOAD_AUDIT').load()
    .filter(F.col('run_id') == run_id)
)
df_audit_summary.groupBy('service', 'status').count().orderBy('service', 'status').show()


+-------+-------+-----+
|service| status|count|
+-------+-------+-----+
|  green|SUCCESS|    1|
| yellow|SUCCESS|    1|
+-------+-------+-----+

